
# 🌲 Otimização de Hiperparâmetros - Random Forest
Este notebook detalha o rigoroso processo de ajuste fino do modelo **Random Forest** para o conjunto de dados do *Diabetes Prediction Challenge*. A estratégia foca na transição de um modelo generalista para um especialista através de buscas estocásticas refinadas.

## 🛠️ Estratégia de Modelagem
* **Pipeline de Dados:** Integração com o preprocessador `v1.2` via `ColumnTransformer`, garantindo o tratamento adequado de variáveis biométricas e categóricas.
* **Otimização de Threshold:** Diferente do padrão (0.5), buscamos ativamente o ponto de corte probabilístico que maximiza a acurácia, adaptando o modelo à distribuição real das classes no teste.
* **Métrica Principal:** Foco em **ROC-AUC** para garantir que o classificador mantenha um alto poder de separação entre as classes em todo o espectro de probabilidades.

## 🔍 Fluxo de Otimização (O Funil)
1.  **Baseline (Referência):** Treinamento da Random Forest com configurações padrão para estabelecer o *benchmark* de desempenho e avaliar a estabilidade inicial via validação cruzada.
2.  **Fase 3.a - Busca Exploratória:** Utilização de `RandomizedSearchCV` com 50 iterações em um espaço de parâmetros propositalmente amplo. O objetivo aqui é identificar quais "zonas" de complexidade (profundidade, número de árvores, critérios de divisão) entregam os melhores resultados.
3.  **Fase 3.b - Refinamento (Refine):** Implementação de uma lógica de Grid Automático. Com base nos melhores resultados da fase anterior, o algoritmo gera um novo espaço de busca estreito (±20% em torno dos melhores valores), permitindo um ajuste fino cirúrgico nos hiperparâmetros.

## 📊 Critérios de Sucesso
* **Validação Estável:** Uso de `StratifiedKFold` para assegurar que a performance do modelo seja consistente e não dependente de um *split* específico dos dados.
* **Capacidade do Ensemble:** Monitoramento de parâmetros de estocasticidade (`max_features`, `max_samples`) para garantir que as árvores do ensemble sejam diversas e robustas ao ruído.
* **Comparativo de Evolução:** Avaliação clara do ganho de performance entre o Baseline, a Busca Exploratória e o Refinamento Final.


In [1]:
import warnings
warnings.filterwarnings("ignore")
import sys

import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
import time

#Random forest
from sklearn.ensemble import RandomForestClassifier

# sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_score, RandomizedSearchCV, StratifiedKFold


from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             average_precision_score,roc_curve)
from sklearn.base import BaseEstimator, TransformerMixin, clone

from scipy.stats import ttest_rel
from scipy.stats import randint, uniform

#hiperparamentros search
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy.stats import randint, uniform,loguniform


# Importações locais
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
import src.preprocess_utils_diab14 as new_utils
sys.modules['src.preprocess_utils_diab'] = new_utils

print("#Processo iniciado em:", time.strftime("%H:%M:%S"))
start_inicial = time.time()


#Processo iniciado em: 20:55:20


### 1. Load Data & Pipeline

In [2]:
BASE = Path.cwd().parent

PP2 = joblib.load(BASE/'src'/'preprocess_diabetes_v1.2.joblib')['preprocessador']

DATA_DIR = BASE/"data"/"raw"
X_train = pd.read_csv(DATA_DIR/"X_train_raw.csv")
X_test  = pd.read_csv(DATA_DIR/"X_test_raw.csv")
y_train = pd.read_csv(DATA_DIR/"y_train_raw.csv").values.ravel()
y_test  = pd.read_csv(DATA_DIR/"y_test_raw.csv").values.ravel()
mtd_scoring='roc_auc'

print("\n#Processo iniciado em:", time.strftime("%H:%M:%S"))


#Processo iniciado em: 20:55:21


## 2.Baseline

In [3]:
print("#Processo iniciado em:", time.strftime("%H:%M:%S"))

# BASELINE
model_base = RandomForestClassifier(random_state=42, n_jobs=4)
pipe_base  = pipe_models(model_base, PP2)
pipe_base.fit(X_train, y_train)

# =====================================================
# 1) Predição
# =====================================================
y_pred_base=pipe_base.predict(X_test)

# =====================================================
# 2) Otimização do threshold de decisão
# =====================================================
# Como classificadores probabilísticos usam threshold padrão de 0.5,
# testamos diferentes valores para verificar se existe um ponto que
# maximize a acurácia no conjunto de teste.
best_th_base,max_acc_base,y_probs_base=best_threshold2(pipe_base, X_train, y_train,X_test,y_test)

# =====================================================
# 3) Desempenho em validação cruzada
# =====================================================
# A validação cruzada utiliza a mesma métrica definida
# no processo de tuning para avaliar a estabilidade do modelo.
cv_s = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores_base = cross_val_score(pipe_base, X_train, y_train,
                                cv=cv_s,
                                scoring=mtd_scoring,
                                n_jobs=-1)

print(f"\n{'='*70}")
print(f" 📍 RESULTADOS BASELINE".center(70))
print(f"{'='*70}")
# =====================================================
# 4) Avaliação por validação cruzada (Treino)
# =====================================================

print("📊 CROSS-VALIDATION")
print(f"   Média {mtd_scoring}:       {cv_scores_base.mean():>15.4f} ± {cv_scores_base.std():.4f}")

# =====================================================
# 5) Avaliação no conjunto de teste
# =====================================================
print(f"\n✅ TEST SET")
print(f"   Padrão (0.5):              {accuracy_score(y_test, y_pred_base):>10.4f}")
print(f"   Otimizado:                 {max_acc_base:>10.4f} (threshold ={best_th_base:>6.3f})")
print(f"   ROC-AUC:                   {roc_auc_score(y_test, y_probs_base):>10.4f}")
print(f"   Avg precision:             {average_precision_score(y_test, y_probs_base):>10.4f}")

# =====================================================
# 6) Hiperparametros utilizados
# =====================================================
print(f"{'='*70}")
print_hiper(model_base.get_params())
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))

#Processo iniciado em: 20:55:21

                         📍 RESULTADOS BASELINE                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.6903 ± 0.0004

✅ TEST SET
   Padrão (0.5):                  0.6624
   Otimizado:                     0.6625 (threshold = 0.490)
   ROC-AUC:                       0.6905
   Avg precision:                 0.7813
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Bootstrap                 : True
• Ccp Alpha                 : 0.0
• Class Weight              : None
• Criterion                 : gini
• Max Depth                 : None
• Max Features              : sqrt
• Max Leaf Nodes            : None
• Max Samples               : None
• Min Impurity Decrease     : 0.0
• Min Samples Leaf          : 1
• Min Samples Split         : 2
• Min Weight Fraction Leaf  : 0.0
• Monotonic Cst             : None
• N Estimators              : 100
• N Jobs                    : 4
• Oob Score                 : False
• Rand

## 3.Buscas por hiperparamentros


### 3.a Exploratorio

In [4]:
print("#Processo Iniciado em:", time.strftime("%H:%M:%S"))
# Exploratório
param_dist_1 = {
    # 1. Capacidade do Ensemble
    'model__n_estimators': randint(100, 3000),
    
    # 2. Complexidade Estrutural da Árvore(Profundidade e tamanho efetivo)
    'model__max_depth': [None, 2, 4, 8, 12, 15,20,25,30],
    'model__max_leaf_nodes': [None, 50, 100, 300, 800, 1500],

    # 3. Critérios de Divisão (Granularidade)
    'model__min_samples_split': randint(2, 40),
    'model__min_samples_leaf': randint(1, 30),
    'model__min_weight_fraction_leaf': uniform(0.0, 0.5),

    # 4. Estocasticidade e Diversidade do Ensemble
    # Equivalente conceitual ao subsample/colsample
    'model__max_features': ['sqrt', 'log2', None, 0.3, 0.5, 0.7],
    'model__bootstrap': [True, False],
    'model__max_samples': [None, 0.5, 0.7, 0.8, 0.9],

    # 5. Critério de Impureza (Bias-inducing)
    'model__criterion': ['gini', 'entropy']
}

cv_s = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
numero_itera=50
search_1 = RandomizedSearchCV(
    pipe_base, param_dist_1,
    n_iter=numero_itera, cv=cv_s,
    scoring=mtd_scoring,
    random_state=42, verbose=1
)

start = time.time()
search_1.fit(X_train, y_train)
end = time.time()

best_1 = search_1.best_estimator_

# -----------------------------------------------------
# 1) Predição
# -----------------------------------------------------
y_pred_1=best_1.predict(X_test)

# -----------------------------------------------------
# 2) Otimização do threshold de decisão
# -----------------------------------------------------
best_th_1,max_acc_1,y_probs_1=best_threshold2(best_1, X_train, y_train,X_test,y_test)

print("\n#Finalizando a validação cruzada em:", time.strftime("%H:%M:%S"))

# -----------------------------------------------------
# 3) Desempenho em validação cruzada
# -----------------------------------------------------
cv_scores_1 = cross_val_score(best_1, X_train, y_train,
                                cv=cv_s,
                                scoring=mtd_scoring,
                                n_jobs=-1) #90s

print(f"\n{'='*70}")
print(f" 📍 RESULTADOS SEARCH 1".center(70))
print(f"{'='*70}")

# -----------------------------------------------------
# 4) Avaliação por validação cruzada (Treino)
# -----------------------------------------------------
print("📊 CROSS-VALIDATION")
print(f"   Média {mtd_scoring}:       {cv_scores_1.mean():>15.4f} ± {cv_scores_1.std():.4f}")

# -----------------------------------------------------
# 5) Avaliação no conjunto de teste
# -----------------------------------------------------
print(f"\n✅ TEST SET")
print(f"   Padrão (0.5):              {accuracy_score(y_test, y_pred_1):>10.4f}")
print(f"   Otimizado:                 {max_acc_1:>10.4f} (threshold ={best_th_1:>6.3f})")
print(f"   ROC-AUC:                   {roc_auc_score(y_test, y_probs_1):>10.4f}")
print(f"   Avg precision:             {average_precision_score(y_test, y_probs_1):>10.4f}")

#-----------------------------------------------------
# 6) Hiperparametros utilizados
#-----------------------------------------------------
print(f"{'='*70}")
print_hiper(search_1.best_params_)
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))
joblib.dump(search_1, 'temp_2.joblib')

#Processo Iniciado em: 20:58:09
Fitting 3 folds for each of 50 candidates, totalling 150 fits

#Finalizando a validação cruzada em: 01:19:09

                         📍 RESULTADOS SEARCH 1                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.6933 ± 0.0006

✅ TEST SET
   Padrão (0.5):                  0.6523
   Otimizado:                     0.6623 (threshold = 0.550)
   ROC-AUC:                       0.6931
   Avg precision:                 0.7881
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Bootstrap                 : True
• Criterion                 : entropy
• Max Depth                 : 12
• Max Features              : sqrt
• Max Leaf Nodes            : 1500
• Max Samples               : 0.8
• Min Samples Leaf          : 5
• Min Samples Split         : 13
• Min Weight Fraction Leaf  : 0.008580550915875118
• N Estimators              : 2621
--------------------------------------------------

#Processo finalizado em: 01:32:1

['temp_2.joblib']

### 3.b Refine

In [12]:
print("#Processo Iniciado em:", time.strftime("%H:%M:%S"))
def generate_refined_grid(best_params):
    """
    Gera um grid refinado automaticamente baseado nos melhores parâmetros
    do Random Search exploratório (Random Forest).
    """
    # ---------------------------
    # 1. n_estimators
    # ---------------------------
    n_best = best_params['model__n_estimators']
    n_min  = int(n_best * 0.8)
    n_max  = int(n_best * 1.2)
    # ---------------------------
    # 2. max_depth (Tratamento de None)
    # ---------------------------
    depth_best = best_params['model__max_depth']
    if depth_best is not None:
        depth_list = [
            max(1, depth_best - 2),
            depth_best,
            depth_best + 2
        ]
    else:
        depth_list = [None, 10, 20]

    # ---------------------------
    # 3. max_leaf_nodes (Tratamento de None)
    # ---------------------------
    leaf_best = best_params.get('model__max_leaf_nodes', None)
    if leaf_best is not None:
        leaf_list = [
            max(10, int(leaf_best * 0.8)),
            leaf_best,
            int(leaf_best * 1.2)
        ]
    else:
        leaf_list = [None, 100, 500]

    # ---------------------------
    # 4. min_samples_split
    # ---------------------------
    mss_best = best_params['model__min_samples_split']
    mss_list = [
        max(2, mss_best - 2),
        mss_best,
        mss_best + 2
    ]

    # ---------------------------
    # 5. min_samples_leaf
    # ---------------------------
    msl_best = best_params['model__min_samples_leaf']
    msl_list = [
        max(1, msl_best - 2),
        msl_best,
        msl_best + 2
    ]

    # ---------------------------
    # 6. min_weight_fraction_leaf
    # ---------------------------
    mwfl_best = best_params.get('model__min_weight_fraction_leaf', 0.0)
    mwfl_st   = max(0.0, mwfl_best * 0.75)
    mwfl_dist = 0.15

    # ---------------------------
    # 7. max_features
    # ---------------------------
    mf_best = best_params['model__max_features']
    if isinstance(mf_best, float):
        mf_list = [
            max(0.1, mf_best - 0.1),
            mf_best,
            min(1.0, mf_best + 0.1)
        ]
    else:
        mf_list = [mf_best]

    # ---------------------------
    # 8. max_samples (Tratamento de None)
    # ---------------------------
    ms_best = best_params.get('model__max_samples', None)
    if ms_best is not None:
        ms_st   = max(0.3, ms_best * 0.8)
        ms_dist = 0.25
    else:
        ms_st   = 0.5
        ms_dist = 0.3

    # ---------------------------
    # PRINTS DIAGNÓSTICOS
    # ---------------------------
    print(f"{'='*30} NOVO GRID DE REFINAMENTO (Random Forest) {'='*30}")
    print(f"🔹 n_estimators              : {n_min} - {n_max}")
    print(f"🔹 max_depth                 : {depth_list}")
    print(f"🔹 max_leaf_nodes            : {leaf_list}")
    print(f"🔹 min_samples_split         : {mss_list}")
    print(f"🔹 min_samples_leaf          : {msl_list}")
    print(f"🔹 min_weight_fraction_leaf  : {mwfl_st:.3f} - {mwfl_st + mwfl_dist:.3f}")
    print(f"🔹 max_features              : {mf_list}")
    print(f"🔹 max_samples               : {ms_st:.3f} - {min(1.0, ms_st + ms_dist):.3f}")
    print(f"{'='*92}\n")

    # ---------------------------
    # GRID FINAL
    # ---------------------------
    refined_grid = {
        'model__n_estimators': randint(n_min, n_max + 1),
        'model__max_depth': depth_list,
        'model__max_leaf_nodes': leaf_list,
        'model__min_samples_split': mss_list,
        'model__min_samples_leaf': msl_list,
        'model__min_weight_fraction_leaf': uniform(mwfl_st, mwfl_dist),
        'model__max_features': mf_list,
        'model__max_samples': uniform(ms_st, ms_dist)
    }

    return refined_grid

# # Aplicação automática
param_dist_2 = generate_refined_grid(search_1.best_params_)

numero_itera = 30
search_2 = RandomizedSearchCV(
    pipe_base, param_dist_2,
    n_iter=numero_itera, cv=cv_s, 
    scoring=mtd_scoring,
    random_state=42, verbose=1
)

start = time.time()
search_2.fit(X_train, y_train)
end = time.time()

best_2 = search_2.best_estimator_
# -----------------------------------------------------
# 1) Predição
# -----------------------------------------------------
y_pred_2=best_2.predict(X_test)

# -----------------------------------------------------
# 2) Otimização do threshold de decisão
# -----------------------------------------------------
best_th_2,max_acc_2,y_probs_2=best_threshold2(best_2, X_train, y_train,X_test,y_test)

print("\n#Finalizando a validação cruzada em:", time.strftime("%H:%M:%S"))

# -----------------------------------------------------
# 3) Desempenho em validação cruzada
# -----------------------------------------------------
cv_scores_2 = cross_val_score(best_2, X_train, y_train,
                                cv=cv_s,
                                scoring=mtd_scoring,
                                n_jobs=-1) #90s

print(f"\n{'='*70}")
print(f" 📍 RESULTADOS SEARCH 2".center(70))
print(f"{'='*70}")

# -----------------------------------------------------
# 4) Avaliação por validação cruzada (Treino)
# -----------------------------------------------------
print("📊 CROSS-VALIDATION")
print(f"   Média {mtd_scoring}:       {cv_scores_2.mean():>15.4f} ± {cv_scores_2.std():.4f}")

# -----------------------------------------------------
# 5) Avaliação no conjunto de teste
# -----------------------------------------------------
print(f"\n✅ TEST SET")
print(f"   Padrão (0.5):              {accuracy_score(y_test, y_pred_2):>10.4f}")
print(f"   Otimizado:                 {max_acc_2:>10.4f} (threshold ={best_th_2:>6.3f})")
print(f"   ROC-AUC:                   {roc_auc_score(y_test, y_probs_2):>10.4f}")
print(f"   Avg precision:             {average_precision_score(y_test, y_probs_2):>10.4f}")

#-----------------------------------------------------
# 6) Hiperparametros utilizados
#-----------------------------------------------------
print(f"{'='*70}")
print_hiper(search_2.best_params_)
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))
joblib.dump(search_2, 'temp_2.joblib')

#Processo Iniciado em: 06:04:20
============================== NOVO GRID DE REFINAMENTO (Random Forest) ==============================
🔹 n_estimators              : 2096 - 3145
🔹 max_depth                 : [10, 12, 14]
🔹 max_leaf_nodes            : [1200, 1500, 1800]
🔹 min_samples_split         : [11, 13, 15]
🔹 min_samples_leaf          : [3, 5, 7]
🔹 min_weight_fraction_leaf  : 0.006 - 0.156
🔹 max_features              : ['sqrt']
🔹 max_samples               : 0.640 - 0.890

Fitting 3 folds for each of 30 candidates, totalling 90 fits

#Finalizando a validação cruzada em: 10:02:51

                         📍 RESULTADOS SEARCH 2                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.6931 ± 0.0005

✅ TEST SET
   Padrão (0.5):                  0.6517
   Otimizado:                     0.6626 (threshold = 0.550)
   ROC-AUC:                       0.6930
   Avg precision:                 0.7880
🎯 Melhores Hiperparâmetros
-------------------------------------------------

['temp_2.joblib']

## 4.Salvando modelos

In [13]:
joblib.dump(search_1.best_estimator_, 'modelo_RF_final_randsearch.'+mtd_scoring+'_v1.2.joblib')
joblib.dump(search_2.best_estimator_, 'modelo_RF_final_refine.'+mtd_scoring+'_v1.2.joblib')
print(f"✅ Arquivo salvo • {time.strftime('%d/%m/%Y-%H:%M:%S')}")

✅ Arquivo salvo • 07/03/2026-11:02:26
